In [ ]:
import os
import random
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import logging
from datasets import Dataset


from sklearn.metrics import accuracy_score
from transformers import TrainingArguments
from transformers import Trainer, TrainerCallback
from transformers.trainer_pt_utils import _get_learning_rate
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer, GenerationConfig, AutoConfig

import torch.distributed as dist


os.environ['TORCH_CUDA_ARCH_LIST'] = '9.0'   # check GPU
# os.environ["TOKENIZERS_PARALLELISM"] = "true"
#os.environ["CUDA_VISIBLE_DEVICES"]= "0"


# ## Seed 고정

In [ ]:


def seed_everything(seed:int = 1004):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # current gpu seed
    torch.cuda.manual_seed_all(seed) # All gpu seed
    torch.backends.cudnn.deterministic = True  
    torch.backends.cudnn.benchmark = False  # True로 하면 gpu에 적합한 알고리즘을 선택함.

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


# ## Config

In [ ]:
MODEL_NAME = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"

In [ ]:


config = AutoConfig.from_pretrained(MODEL_NAME,
                                   trust_remote_code=True)  

print(config)

In [ ]:


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,)  # 토크나이저는 학습되어 있는 것으로
tokenizer.model_max_length=8192

In [ ]:


print(tokenizer)

In [ ]:


print(tokenizer.eos_token_id, tokenizer.pad_token_id)

In [ ]:


print(tokenizer("Hello I'm World!")), print(tokenizer.decode(tokenizer.encode("Hello I'm World!")))

In [ ]:


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation = "flash_attention_2"
    #device_map={"": 0}
)

model.config.use_cache = False

print(next(model.parameters()).dtype)
print(model)

In [ ]:
print(model.num_parameters())


# ## 데이터 준비

In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from trl import SFTConfig
from collections import defaultdict
from collections.abc import Mapping
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Optional, TypeVar, Union
import torch
import torch.nn as nn
import transformers
from accelerate import PartialState, logging
from datasets import Dataset, IterableDataset
from transformers import (
    AutoConfig,
    AutoProcessor,
    BaseImageProcessor,
    DataCollator,
    FeatureExtractionMixin,
    PreTrainedModel,
    PreTrainedTokenizerBase,
    ProcessorMixin,
    Trainer,
    TrainingArguments,
    is_wandb_available,
)
from transformers.data.data_collator import DataCollatorMixin
from transformers.trainer_callback import TrainerCallback
from transformers.trainer_utils import EvalPrediction
from transformers.utils import is_peft_available
from trl.data_utils import (
    apply_chat_template,
    is_conversational,
    is_conversational_from_value,
    maybe_convert_to_chatml,
    pack_dataset,
    prepare_multimodal_messages,
    truncate_dataset,
)

training_args = SFTConfig(
    output_dir="/home/work/Korean_Spoken_model/EXAONE3.5_7.8B_packed_SFT/",
    logging_dir= "/home/work/Korean_Spoken_model/EXAONE3.5_7.8B_packed_SFT_LOG/",
    deepspeed = "/home/work/Korean_Spoken_model/ds_config.json",  # DeepSpeed Zero 2 no offload optim
    num_train_epochs=3,
    learning_rate = 1e-4,
    bf16 =True,
    optim = "adamw_torch_fused",
    per_device_train_batch_size=2,
    gradient_checkpointing=False, 
    torch_compile=True,
    gradient_accumulation_steps = 2,
    logging_strategy = "steps",
    save_strategy = "epoch",
    lr_scheduler_type = "linear",
    warmup_ratio=0.01,
    weight_decay=0.01,
  #  save_steps=10000,
    logging_steps=10000,
    save_total_limit =3,
    report_to="tensorboard",
    dataset_text_field = "Text",
    packing = True,
    ddp_timeout =30000
)


def _prepare_dataset(
        dataset: Union[Dataset, IterableDataset],
        processing_class: Union[PreTrainedTokenizerBase, BaseImageProcessor, FeatureExtractionMixin, ProcessorMixin],
        args: SFTConfig,
        packing: bool,
        formatting_func: Optional[Callable[[dict], str]],
        dataset_name: str,
    ) -> Union[Dataset, IterableDataset]:
        # Tabular backends like Arrow/Parquet insert `None` for mismatched keys in nested structures. Clean them from
        # sampled data.
        if isinstance(dataset, Dataset):  # IterableDataset does not support `with_transform`
            dataset = dataset.with_transform(remove_none_values)

        # If the dataset is already preprocessed (tokenized), skip the processing steps.
        column_names = list(next(iter(dataset)).keys())
        is_processed = "input_ids" in column_names

        # Build the kwargs for the `map` function
        map_kwargs = {}
        if isinstance(dataset, Dataset):  # IterableDataset does not support num_proc
            map_kwargs["num_proc"] = args.dataset_num_proc

        with PartialState().main_process_first():
            # Apply the formatting function if any
            if formatting_func is not None and is_processed:
                logger.warning(
                    "You passed a dataset that is already processed (contains an `input_ids` field) together with a "
                    "formatting function. Therefore `formatting_func` will be ignored. Either remove the "
                    "`formatting_func` or pass a dataset that is not already processed.",
                )

            if formatting_func is not None and not is_processed:
                if isinstance(dataset, Dataset):  # `IterableDataset.map` does not support `desc`
                    map_kwargs["desc"] = f"Applying formatting function to {dataset_name} dataset"

                def _func(example):
                    return {"text": formatting_func(example)}

                dataset = dataset.map(_func, batched=False, **map_kwargs)

            if not is_processed:
                # Convert the dataset to ChatML if needed
                first_example = next(iter(dataset))
                if is_conversational_from_value(first_example):
                    if isinstance(dataset, Dataset):  # `IterableDataset.map` does not support `desc`
                        map_kwargs["desc"] = f"Converting {dataset_name} dataset to ChatML"
                    column_names = next(iter(dataset)).keys()
                    dataset = dataset.map(
                        maybe_convert_to_chatml,
                        remove_columns="conversations" if "conversations" in column_names else None,
                        **map_kwargs,
                    )

                # Apply the chat template if needed
                first_example = next(iter(dataset))
                if not is_conversational(first_example):
                    if isinstance(dataset, Dataset):  # `IterableDataset.map` does not support `desc`
                        map_kwargs["desc"] = f"Adding EOS to {dataset_name} dataset"

                    def add_eos(example, eos_token):
                        if "text" in example and not example["text"].endswith(eos_token):  # language modeling case
                            example["text"] = example["text"] + eos_token
                        elif "completion" in example and not example["completion"].endswith(eos_token):
                            example["completion"] = example["completion"] + eos_token
                        return example

                    dataset = dataset.map(
                        add_eos,
                        fn_kwargs={"eos_token": processing_class.eos_token},
                        remove_columns="messages" if "messages" in column_names else None,  # renamed to "text"
                        **map_kwargs,
                    )

                # Tokenize the dataset
                if isinstance(dataset, Dataset):  # `IterableDataset.map` does not support `desc`
                    map_kwargs["desc"] = f"Tokenizing {dataset_name} dataset"

                def tokenize(example, processing_class, dataset_text_field, assistant_only_loss):
                    if "prompt" in example:  # prompt-completion case
                        output = {}
                        if is_conversational(example):
                            prompt_ids = processing_class.apply_chat_template(
                                example["prompt"],
                                tools=example.get("tools"),
                                **example.get("chat_template_kwargs", {}),
                            )
                            prompt_completion_processed = processing_class.apply_chat_template(
                                example["prompt"] + example["completion"],
                                return_dict=True,
                                return_assistant_tokens_mask=assistant_only_loss,
                                tools=example.get("tools"),
                                **example.get("chat_template_kwargs", {}),
                            )
                            prompt_completion_ids = prompt_completion_processed["input_ids"]
                            if "assistant_masks" in prompt_completion_processed:
                                output["assistant_masks"] = prompt_completion_processed["assistant_masks"]
                        else:
                            prompt_ids = processing_class(text=example["prompt"])["input_ids"]
                            prompt_completion_ids = processing_class(text=example["prompt"] + example["completion"])[
                                "input_ids"
                            ]

                        # Check if the tokenized prompt starts with the tokenized prompt+completion
                        if not prompt_completion_ids[: len(prompt_ids)] == prompt_ids:
                            logger.warning(
                                "Mismatch between tokenized prompt and the start of tokenized prompt+completion. "
                                "This may be due to unexpected tokenizer behavior, whitespace issues, or special "
                                "token handling. Verify that the tokenizer is processing text consistently."
                            )

                        # Create a completion mask
                        completion_mask = [0] * len(prompt_ids) + [1] * (len(prompt_completion_ids) - len(prompt_ids))
                        output["input_ids"] = prompt_completion_ids
                        output["completion_mask"] = completion_mask

                    else:  # language modeling case
                        if is_conversational(example):
                            processed = processing_class.apply_chat_template(
                                example["messages"],
                                return_dict=True,
                                return_assistant_tokens_mask=assistant_only_loss,
                                tools=example.get("tools"),
                                **example.get("chat_template_kwargs", {}),
                            )
                            if "assistant_masks" in processed and 1 not in processed["assistant_masks"]:
                                raise RuntimeError(
                                    "You're using `assistant_only_loss=True`, but at least one example has no "
                                    "assistant tokens. This usually means the tokenizer's chat template doesn't "
                                    "generate assistant masks — it may be missing the `{% generation %}` keyword. Please "
                                    "check the template and ensure it's correctly configured to support assistant "
                                    "masking."
                                )
                            output = {k: processed[k] for k in ("input_ids", "assistant_masks") if k in processed}
                        else:
                            output = {"input_ids": processing_class(text=example[dataset_text_field])["input_ids"]}
                    return output

                dataset = dataset.map(
                    tokenize,
                    fn_kwargs={
                        "processing_class": processing_class,
                        "dataset_text_field": args.dataset_text_field,
                        "assistant_only_loss": args.assistant_only_loss,
                    },
                    **map_kwargs,
                )

            # Pack or truncate
            if packing:
                if args.max_length is None:
                    raise ValueError("When packing is enabled, `max_length` can't be `None`.")
                if isinstance(dataset, Dataset):  # `IterableDataset.map` does not support `desc`
                    map_kwargs["desc"] = f"Packing {dataset_name} dataset"

                columns = ["input_ids"]
                if "completion_mask" in dataset.column_names:
                    columns.append("completion_mask")
                if "assistant_masks" in dataset.column_names:
                    columns.append("assistant_masks")

                dataset = dataset.select_columns(columns)

                # Packing adds new column "seq_lengths" needed for document aware FlashAttention
                dataset = pack_dataset(dataset, args.max_length, args.packing_strategy, map_kwargs)
            elif args.max_length is not None:
                if isinstance(dataset, Dataset):  # `IterableDataset.map` does not support `desc`
                    map_kwargs["desc"] = f"Truncating {dataset_name} dataset"
                dataset = truncate_dataset(dataset, args.max_length, map_kwargs)
            # For Liger kernel, ensure only the essential columns
            if args.use_liger_kernel:
                collator_expected_keys = {"input_ids", "seq_lengths", "completion_mask", "assistant_masks"}
                dataset = dataset.select_columns(collator_expected_keys.intersection(dataset.column_names))

        return dataset

In [ ]:
# from trl.trainer.sft_trainer import remove_none_values

# packed_data = _prepare_dataset(dataset = train_data, 
#                                processing_class = tokenizer, 
#                                args = training_args, 
#                                packing = training_args.packing, 
#                                formatting_func = None,
#                                dataset_name = "train")

In [ ]:
import pickle

# # save
# with open('/home/work/Korean_Spoken/SFTpacked_whole_spoken+written_list.pkl', 'wb') as f:
#     pickle.dump(packed_data, f)
    
# load
with open('/home/work/Korean_Spoken/SFTpacked_whole_spoken+written_list.pkl', 'rb') as f4:
    train_data = pickle.load(f4)
    
print("Data Loading Complete.")

In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from trl import SFTConfig


training_args = SFTConfig(
    output_dir="/home/work/Korean_Spoken_model/EXAONE3.5_7.8B_packed_SFT/",
    logging_dir= "/home/work/Korean_Spoken_model/EXAONE3.5_7.8B_packed_SFT_LOG/",
    deepspeed = "/home/work/Korean_Spoken_model/ds_config.json",  # DeepSpeed Zero 2 no offload optim
    num_train_epochs=3,
    learning_rate = 1e-4,
    bf16 =True,
    optim = "adamw_torch_fused",
    per_device_train_batch_size=8,
    gradient_checkpointing=False, 
    torch_compile=True,
    gradient_accumulation_steps = 2,
    logging_strategy = "steps",
    save_strategy = "epoch",
    lr_scheduler_type = "linear",
    warmup_ratio=0.01,
    weight_decay=0.01,
  #  save_steps=10000,
    logging_steps=10000,
    save_total_limit =3,
    report_to="tensorboard",
    dataset_text_field = "Text",
    packing = True,
    ddp_timeout =30000,
    dataset_kwargs  = {"skip_prepare_dataset" : True}
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    processing_class=tokenizer,
)

In [ ]:


print("Done")

In [ ]:


#trainer.add_callback(CustomCallback(trainer))
trainer.train()

In [ ]:


trainer.save_model("/home/work/Korean_Spoken_model/Final_LGEXAONE3.5_7.8B_packed_SFT")

